[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/76_nano_gpt_solution.ipynb)

# Solution: Minimal GPT (nanoGPT) — Model + Train + Generate

Reference solution — a compact decoder-only Transformer.

## 解析

**结论：GPT 就是「Embedding → N 个 pre-norm 因果自注意力 Block → LayerNorm → 线性头」的自回归语言模型；训练用 shifted cross-entropy，推理用逐 token 贪心解码。**

### 1. 模型结构（GPT-3 / nanoGPT 核心）
- **两套 Embedding**：`tok_emb`（token → 向量）与 `pos_emb`（位置 → 向量，GPT 用**可学习**位置编码而非 sin/cos）。输入表示 `x = tok_emb(idx) + pos_emb(pos)`。
- **Block ×N**，采用 **pre-norm**（LN 放在子层之前，GPT-2 起的标准写法，训练更稳）：
  - `x = x + attn(ln1(x))`
  - `x = x + mlp(ln2(x))`
- **因果自注意力**：QKV 一次性投影后拆成多头，注意力打分 `softmax(QKᵀ/√d)`，关键是用下三角 mask 把「未来位置」置为 `-inf`，保证位置 `t` 只能看到 `<= t`。
- **MLP**：`Linear(d, 4d) → GELU → Linear(4d, d)`，4× 扩张是 Transformer 惯例。
- 末尾 `ln_f` + `lm_head`（`Linear(d, vocab)`）输出每个位置对下一 token 的 logits。

### 2. 训练
语言模型目标是「预测下一个 token」。`forward(idx, targets)` 里 `targets` 通常是 `idx` 右移一位。损失把 `logits` 展平成 `(B·T, vocab)`、`targets` 展平成 `(B·T,)`，做一次 `F.cross_entropy`。随机初始化时每个 token 近似均匀分布，故初始 loss ≈ `ln(vocab_size)`——这也是测试里 `|loss - ln(17)| < 1.5` 的由来。训练循环就是标准的 `zero_grad → forward → backward → step`。

### 3. 推理（generate）
自回归解码：每步用当前序列前向，取**最后一个位置**的 logits，`argmax` 得到下一 token（贪心/greedy，确定性），拼接到序列末尾，重复 `max_new_tokens` 次。必须做 **context cropping**：因为位置编码只到 `block_size`，输入超长时只保留最后 `block_size` 个 token（`idx[:, -block_size:]`），否则 `pos_emb` 越界。若要做采样/温度/top-k，只需把 `argmax` 换成对 `softmax(logits/T)` 的多项式采样。

### 4. 因果性为何重要
训练时我们让模型在一个前向里对所有位置**并行**预测各自的下一 token；只有因果 mask 才能保证位置 `t` 的预测没有偷看到答案（`t+1..`）。测试通过「扰动后面的 token，检查前面的 logits 不变」来验证这一点。

### 复杂度
单次前向 `O(B · n_layer · (T² · d + T · d²))`——`T²` 来自注意力矩阵，`d²` 来自线性投影。`generate` 每步都对整段前缀重新前向，朴素实现为 `O(Σ_t t²)`；工程上会加 **KV-cache** 把每步降到 `O(T · d)`（本题为保持精简未要求）。

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# ✅ SOLUTION

class _CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.n_embd = n_embd
        self.qkv = nn.Linear(n_embd, 3 * n_embd)      # fused Q, K, V projection
        self.proj = nn.Linear(n_embd, n_embd)
        # lower-triangular causal mask, registered as a buffer (not a parameter)
        self.register_buffer(
            'mask', torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        )

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(self.n_embd, dim=2)
        hs = C // self.n_head
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)   # (B, nh, T, hs)
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / (hs ** 0.5))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))  # causal
        att = F.softmax(att, dim=-1)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)


class _Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = _CausalSelfAttention(n_embd, n_head, block_size)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # pre-norm residual
        x = x + self.mlp(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer, n_head, n_embd):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)   # learnable positional embedding
        self.blocks = nn.ModuleList([_Block(n_embd, n_head, block_size) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None, :, :]
        for blk in self.blocks:
            x = blk(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]         # crop context to block_size
            logits, _ = self(idx_cond)
            next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)   # greedy
            idx = torch.cat([idx, next_id], dim=1)
        return idx

In [ ]:
# Demo: overfit a tiny batch, then generate
torch.manual_seed(0)
model = GPT(vocab_size=13, block_size=16, n_layer=2, n_head=2, n_embd=32)
idx = torch.randint(0, 13, (2, 16))
targets = torch.randint(0, 13, (2, 16))
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for step in range(50):
    opt.zero_grad()
    _, loss = model(idx, targets)
    loss.backward()
    opt.step()
print('final loss:', round(loss.item(), 4))
model.eval()
print('generate:', model.generate(idx[:, :3], max_new_tokens=5).shape)

In [ ]:
from torch_judge import check
check('nano_gpt')